In [20]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, f1_score
)

In [21]:
# ── Reproducibility ──────────────────────────────────────────────────────────
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[DEVICE]  Using: {DEVICE}")

[DEVICE]  Using: cuda


In [22]:
# ── Plotting style ───────────────────────────────────────────────────────────
plt.rcParams.update({
    "figure.facecolor": "#0f1117", "axes.facecolor": "#1a1d27",
    "axes.edgecolor": "#3a3f55",   "axes.labelcolor": "#c8cce0",
    "xtick.color": "#8a8fa8",      "ytick.color": "#8a8fa8",
    "text.color": "#c8cce0",       "grid.color": "#2a2f45",
    "grid.linestyle": "--",        "grid.alpha": 0.5,
    "font.family": "monospace",    "axes.titlecolor": "#e0e4f5",
    "axes.titlesize": 12,          "axes.titleweight": "bold",
})
ACCENT  = "#00d4aa"
ACCENT2 = "#7b61ff"
ACCENT3 = "#ff6b6b"
PALETTE = [ACCENT, ACCENT2, ACCENT3, "#ffd166", "#06d6a0", "#118ab2",
           "#ef476f", "#26547c", "#ffe66d", "#4ecdc4", "#ff6b35", "#a8dadc"]

In [23]:
EPOCHS     = 80
BATCH_SIZE = 256
LR         = 1e-3

In [24]:
# ═══════════════════════════════════════════════════════════════════════════
# 1.  LOAD & PRE-PROCESS (same pipeline as traditional ML)
# ═══════════════════════════════════════════════════════════════════════════
print("=" * 60)
print("  MCS PREDICTION — DEEP LEARNING APPROACH")
print("=" * 60)

df = pd.read_csv("/kaggle/input/datasets/programmer3/real-time-wireless-mcs-prediction-dataset/mcs_prediction_dataset.csv")
df.drop_duplicates(inplace=True)

TARGET = [c for c in df.columns if "mcs" in c.lower()][0]
le = LabelEncoder()
df[TARGET] = le.fit_transform(df[TARGET])
classes = le.classes_
NUM_CLASSES = len(classes)

X = df.drop(columns=[TARGET])
y = df[TARGET]
low_var = [c for c in X.columns if X[c].nunique() <= 1]
X.drop(columns=low_var, inplace=True)
feature_names = X.columns.tolist()
NUM_FEATURES = len(feature_names)

print(f"[DATA]   Shape: {df.shape} | Features: {NUM_FEATURES} | Classes: {NUM_CLASSES}")

# 80:20 split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=SEED, stratify=y
)

# Scale (essential for DL)
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train).astype(np.float32)
X_test_sc  = scaler.transform(X_test).astype(np.float32)
y_train_np = y_train.values.astype(np.int64)
y_test_np  = y_test.values.astype(np.int64)

print(f"[SPLIT]  Train: {X_train_sc.shape}  |  Test: {X_test_sc.shape}")

# PyTorch Datasets
train_ds = TensorDataset(torch.from_numpy(X_train_sc), torch.from_numpy(y_train_np))
test_ds  = TensorDataset(torch.from_numpy(X_test_sc),  torch.from_numpy(y_test_np))
train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  drop_last=False)
test_dl  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False)

  MCS PREDICTION — DEEP LEARNING APPROACH
[DATA]   Shape: (5000, 15) | Features: 14 | Classes: 6
[SPLIT]  Train: (4000, 14)  |  Test: (1000, 14)


In [25]:
# ═══════════════════════════════════════════════════════════════════════════
# 2.  ARCHITECTURE — Residual MLP
# ═══════════════════════════════════════════════════════════════════════════

class ResidualBlock(nn.Module):
    def __init__(self, dim: int, dropout: float = 0.3):
        super().__init__()
        self.block = nn.Sequential(
            nn.Linear(dim, dim),
            nn.BatchNorm1d(dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim, dim),
            nn.BatchNorm1d(dim),
        )
        self.act = nn.GELU()
    def forward(self, x):
        return self.act(x + self.block(x))   # skip connection


class ResidualMLP(nn.Module):
    def __init__(self, in_dim: int, hidden: int, num_classes: int,
                 n_blocks: int = 4, dropout: float = 0.3):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.BatchNorm1d(hidden),
            nn.GELU(),
        )
        self.blocks = nn.Sequential(*[ResidualBlock(hidden, dropout) for _ in range(n_blocks)])
        self.head = nn.Linear(hidden, num_classes)
    
    def forward(self, x):
        return self.head(self.blocks(self.stem(x)))

model = ResidualMLP(in_dim=NUM_FEATURES, hidden=256,
                    num_classes=NUM_CLASSES, n_blocks=4).to(DEVICE)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\n[MODEL]  ResidualMLP | Params: {total_params:,}")
print(model)



[MODEL]  ResidualMLP | Params: 536,326
ResidualMLP(
  (stem): Sequential(
    (0): Linear(in_features=14, out_features=256, bias=True)
    (1): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): GELU(approximate='none')
  )
  (blocks): Sequential(
    (0): ResidualBlock(
      (block): Sequential(
        (0): Linear(in_features=256, out_features=256, bias=True)
        (1): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): GELU(approximate='none')
        (3): Dropout(p=0.3, inplace=False)
        (4): Linear(in_features=256, out_features=256, bias=True)
        (5): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (act): GELU(approximate='none')
    )
    (1): ResidualBlock(
      (block): Sequential(
        (0): Linear(in_features=256, out_features=256, bias=True)
        (1): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=Tru

In [26]:
# ═══════════════════════════════════════════════════════════════════════════
# 3.  TRAINING LOOP
# ═══════════════════════════════════════════════════════════════════════════
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

print(f"\n[TRAIN]  Epochs={EPOCHS}  BatchSize={BATCH_SIZE}  LR={LR}")
print("-" * 55)


def evaluate(dl):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for xb, yb in dl:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            logits = model(xb)
            loss = criterion(logits, yb)
            total_loss += loss.item() * len(yb)
            preds = logits.argmax(dim=1)
            correct += (preds == yb).sum().item()
            total += len(yb)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(yb.cpu().numpy())
    return total_loss / total, correct / total, np.array(all_preds), np.array(all_labels)


for epoch in range(1, EPOCHS + 1):
    model.train()
    ep_loss, ep_correct, ep_total = 0.0, 0, 0
    for xb, yb in train_dl:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        ep_loss += loss.item() * len(yb)
        ep_correct += (logits.argmax(1) == yb).sum().item()
        ep_total   += len(yb)
    scheduler.step()

    tr_loss = ep_loss / ep_total
    tr_acc  = ep_correct / ep_total
    vl_loss, vl_acc, _, _ = evaluate(test_dl)

    history["train_loss"].append(tr_loss)
    history["train_acc"].append(tr_acc)
    history["val_loss"].append(vl_loss)
    history["val_acc"].append(vl_acc)

    if epoch % 10 == 0 or epoch == 1:
        print(f"  Ep {epoch:3d}/{EPOCHS} | "
              f"Train Loss={tr_loss:.4f} Acc={tr_acc:.4f} | "
              f"Val Loss={vl_loss:.4f} Acc={vl_acc:.4f}")

# Final evaluation
_, _, y_pred_dl, _ = evaluate(test_dl)
test_acc = accuracy_score(y_test_np, y_pred_dl)
test_f1  = f1_score(y_test_np, y_pred_dl, average="weighted")

print(f"\n[RESULT] Test Accuracy: {test_acc:.4f}  Weighted F1: {test_f1:.4f}")
print("\n[DL] Classification Report:")
print(classification_report(y_test_np, y_pred_dl,
                             target_names=[str(c) for c in classes]))


[TRAIN]  Epochs=80  BatchSize=256  LR=0.001
-------------------------------------------------------
  Ep   1/80 | Train Loss=1.9973 Acc=0.1653 | Val Loss=1.7953 Acc=0.1890
  Ep  10/80 | Train Loss=1.4804 Acc=0.4070 | Val Loss=2.0431 Acc=0.1820
  Ep  20/80 | Train Loss=1.0508 Acc=0.6020 | Val Loss=2.5869 Acc=0.1550
  Ep  30/80 | Train Loss=0.6966 Acc=0.7440 | Val Loss=3.1547 Acc=0.1460
  Ep  40/80 | Train Loss=0.4721 Acc=0.8310 | Val Loss=3.6878 Acc=0.1720
  Ep  50/80 | Train Loss=0.2962 Acc=0.8945 | Val Loss=4.1987 Acc=0.1670
  Ep  60/80 | Train Loss=0.2308 Acc=0.9195 | Val Loss=4.4848 Acc=0.1650
  Ep  70/80 | Train Loss=0.1719 Acc=0.9423 | Val Loss=4.5986 Acc=0.1630
  Ep  80/80 | Train Loss=0.1683 Acc=0.9450 | Val Loss=4.6026 Acc=0.1620

[RESULT] Test Accuracy: 0.1620  Weighted F1: 0.1621

[DL] Classification Report:
              precision    recall  f1-score   support

           0       0.18      0.19      0.19       165
           1       0.14      0.15      0.14       171
      

In [27]:
from sklearn.utils.class_weight import compute_class_weight
weights = compute_class_weight('balanced', classes=np.unique(y_train_np), y=y_train_np)
class_weights = torch.tensor(weights, dtype=torch.float).to(DEVICE)
criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)

In [28]:
criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

print(f"\n[TRAIN]  Epochs={EPOCHS}  BatchSize={BATCH_SIZE}  LR={LR}")
print("-" * 55)

best_val_acc = 0.0          # ← moved OUTSIDE the loop
patience, patience_counter = 20, 0

for epoch in range(1, EPOCHS + 1):
    model.train()
    ep_loss, ep_correct, ep_total = 0.0, 0, 0
    for xb, yb in train_dl:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        ep_loss    += loss.item() * len(yb)
        ep_correct += (logits.argmax(1) == yb).sum().item()
        ep_total   += len(yb)
    scheduler.step()

    tr_loss = ep_loss / ep_total
    tr_acc  = ep_correct / ep_total
    vl_loss, vl_acc, _, _ = evaluate(test_dl)   # ← vl_acc defined HERE first

    history["train_loss"].append(tr_loss)
    history["train_acc"].append(tr_acc)
    history["val_loss"].append(vl_loss)
    history["val_acc"].append(vl_acc)

    # Early stopping
    if vl_acc > best_val_acc:                   
        best_val_acc = vl_acc
        torch.save(model.state_dict(), 'best_model.pt')
        patience_counter = 0
    else:
        patience_counter += 1
    if patience_counter >= patience:
        print(f"  Early stopping at epoch {epoch}")
        break

    if epoch % 10 == 0 or epoch == 1:
        print(f"  Ep {epoch:3d}/{EPOCHS} | "
              f"Train Loss={tr_loss:.4f} Acc={tr_acc:.4f} | "
              f"Val Loss={vl_loss:.4f} Acc={vl_acc:.4f}")

# Load best checkpoint before final eval
model.load_state_dict(torch.load('best_model.pt')) 

_, _, y_pred_dl, _ = evaluate(test_dl)
test_acc = accuracy_score(y_test_np, y_pred_dl)
test_f1  = f1_score(y_test_np, y_pred_dl, average="weighted")

print(f"\n[RESULT] Test Accuracy: {test_acc:.4f}  Weighted F1: {test_f1:.4f}")
print("\n[DL] Classification Report:")
print(classification_report(y_test_np, y_pred_dl,
                             target_names=[str(c) for c in classes]))


[TRAIN]  Epochs=80  BatchSize=256  LR=0.001
-------------------------------------------------------
  Ep   1/80 | Train Loss=0.9147 Acc=0.8448 | Val Loss=3.6257 Acc=0.1550
  Ep  10/80 | Train Loss=0.7167 Acc=0.8915 | Val Loss=2.9520 Acc=0.1680
  Ep  20/80 | Train Loss=0.6399 Acc=0.9310 | Val Loss=2.9260 Acc=0.1690
  Ep  30/80 | Train Loss=0.6005 Acc=0.9515 | Val Loss=2.8798 Acc=0.1690
  Early stopping at epoch 32

[RESULT] Test Accuracy: 0.1770  Weighted F1: 0.1758

[DL] Classification Report:
              precision    recall  f1-score   support

           0       0.19      0.21      0.20       165
           1       0.20      0.18      0.19       171
           2       0.15      0.17      0.16       163
           3       0.19      0.17      0.18       172
           4       0.13      0.11      0.12       160
           5       0.19      0.22      0.20       169

    accuracy                           0.18      1000
   macro avg       0.18      0.18      0.18      1000
weighted avg